# Seq2Seq 실습 - 한국어 번역기 만들기

[과제명] Seq2Seq 실습
[날짜] 2026-09-08
[목표] 프로젝트(과제) : Seq2seq으로 한국어 번역기 만들기
[사용 기술] PyTorch

**이 노트북은 실제 로컬 `123.py`(2026-09-08 기준 최신본)를 그대로 셀 단위로 옮긴 것입니다.**
로직/주석/변수명 전부 원본 그대로이고, 딱 한 줄만 고쳤습니다: import 구역에 `import random`을 추가했습니다.
(`AttnDecoder.forward`에서 `random.random()`을 쓰는데 원본에는 이 import가 빠져 있어서, 지금 상태로 돌리면
학습 루프 첫 배치에서 `NameError: name 'random' is not defined`로 바로 죽습니다. 그 외에는 아무것도 안 건드렸습니다.)


## 0. 라이브러리 import

In [1]:
from math import e
import pandas
import matplotlib

import sys
import os
import re
import urllib.request
import zipfile
import sentencepiece as spm
import pandas as pd
import random  # <-- 추가: AttnDecoder.forward()의 teacher forcing에서 random.random()을 쓰는데 원본엔 이 import가 없었음 (없으면 NameError)

import numpy as np
import torch
from torch._dynamo.variables import nn_module
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

# [주의] Windows 콘솔 기본 인코딩(cp949)에서는 "✅" 같은 이모지 print 시
# UnicodeEncodeError가 남. stdout/stderr를 utf-8로 강제 재설정해서 방지.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")


## Step 1. 데이터 로드 및 확인

In [2]:
# 학습 파일 위치한 경로
# os.path.expanduser 보다 path가 직관적이라 해당 명령어 사용함.
dataset_dir = Path("~/Desktop/아이펠 모음/Seq2seq/Seq2seq_ko-en/korean-english-park.train").expanduser()

# print(f"1. 설정된 경로 (절대 경로): {os.path.abspath(dataset_dir)}")

# 원문 파일과 번역본 파일의 실제 파일명 지정
src_file_path = dataset_dir / "korean-english-park.train.ko"
tar_file_path = dataset_dir / "korean-english-park.train.en"

# 파일 존재 여부 확인 및 읽기
if src_file_path.exists() and tar_file_path.exists():
    print("✅ 두 파일을 모두 성공적으로 찾았습니다!")

    # 원문 읽기
    with open(src_file_path, "r", encoding="utf-8") as f:
        src_data = [line.strip() for line in f.readlines()]

    # 번역본 읽기
    with open(tar_file_path, "r", encoding="utf-8") as f:
        tar_data = [line.strip() for line in f.readlines()]

    print(f"원문 문장 수: {len(src_data)}개")
    print(f"번역본 문장 수: {len(tar_data)}개")

    # 샘플 출력
    print("\n--- [데이터 샘플] ---")
    print("원문(KO):", src_data[0])
    print("번역(EN):", tar_data[0])

else:
    print("❌ 파일을 찾을 수 없습니다. 파일 이름을 확인해 주세요.")
    print("현재 폴더 내 실제 파일 목록:", os.listdir(dataset_dir) if dataset_dir.exists() else "폴더 없음")


✅ 두 파일을 모두 성공적으로 찾았습니다!
원문 문장 수: 94123개
번역본 문장 수: 94123개

--- [데이터 샘플] ---
원문(KO): 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
번역(EN): Much of personal computing is about "can you top this?"


## Step 2. 전처리 함수 및 코퍼스 구축

In [3]:
# 전처리 함수 (공백, 기호 제거)
def preprocess_korean(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^가-힣0-9?.!, ]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence


# 영어 기반 전처리
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-zA-Z0-9?.!, ]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence


In [4]:
# zip()으로 한국어-영어 쌍 튜플 만들기, 원본 기준 중복 제거
cleaned_corpus = set(zip(src_data, tar_data))
cleaned_corpus = {(preprocess_korean(src), preprocess_sentence(tar)) # 변수 덮어씀
                   for src, tar in cleaned_corpus}


In [5]:
# coupus 각각 구축 (40어절 이하)
ko_corpus = []
en_corpus = []

for src, tar in cleaned_corpus:
    if len(src.split()) <= 40 and len(tar.split()) <= 40:
        ko_corpus.append(src)
        en_corpus.append(tar)  # 이거 src -> tar 못찾아서 30분동안 개고생함

assert len(ko_corpus) == len(en_corpus)
print(len(ko_corpus), len(en_corpus))


71445 71445


In [6]:
# 파일 경로때문에 오류 발생하여 경로 직접 지정
seq2seq_dir = './Seq2seq' #상대 경로
os.makedirs(seq2seq_dir, exist_ok=True)

# 전처리한 텍스트 파일 생성
ko_corpus_path = os.path.join(seq2seq_dir, 'ko_corpus.txt')
en_corpus_path = os.path.join(seq2seq_dir, 'en_corpus.txt')

with open(ko_corpus_path, 'w', encoding = 'utf-8') as f:
    for line in ko_corpus:
        f.write(line + '\n')

with open(en_corpus_path, 'w', encoding = 'utf-8') as f:
    for line in en_corpus:
        f.write(line + '\n')


## Step 3. Tokenization (SentencePiece)

In [7]:
VOCAB_SIZE = 32000   # 코퍼스 크기에 맞게 (에러 나면 메시지 보고 조정)

spm.SentencePieceTrainer.train(
    input=ko_corpus_path,
    model_prefix=os.path.join(seq2seq_dir, 'ko_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='unigram',
    pad_id=0, bos_id=1, eos_id=2, unk_id=3,
    normalization_rule_name="identity"
    # 각각 채움, 시작, 끝, 처음 보는 토큰을 의미
    # 찐빠나서 normalization_rule_name="identity" 추가
    # 이거 디버깅하느라 20분 소요함 ;;
)

spm.SentencePieceTrainer.train(
    input=en_corpus_path,
    model_prefix=os.path.join(seq2seq_dir, 'en_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='unigram',
    pad_id=0, bos_id=1, eos_id=2, unk_id=3,
    normalization_rule_name="identity"
)

# 학습된 토크나이저 불러오기
encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load(os.path.join(seq2seq_dir, 'ko_spm.model'))

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load(os.path.join(seq2seq_dir, 'en_spm.model'))


True

## Step 3-1. Dataset / DataLoader

In [10]:
# 상속받을 데이터셋 클래스 정의
# __len__, __getitem__을 직접 구현해야 DataLoader가 이 객체를 배치 단위로 순회할 수 있음.
class TranslationDataset(Dataset):
    def __init__(self, src_corpus, trg_corpus, encoder_tokenizer, decoder_tokenizer, max_len):
        self.src_corpus = src_corpus       # ko_corpus
        self.trg_corpus = trg_corpus       # en_corpus
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len

        # 하드코딩 대신 토크나이저에서 직접 가져옴 (vocab이 바뀌어도 코드 깨지지x)
        self.pad_id = decoder_tokenizer.pad_id()
        self.bos_id = decoder_tokenizer.bos_id()
        self.eos_id = decoder_tokenizer.eos_id()

    # 데이터셋 전체 샘플 개수를 반환
    def __len__(self):
        return len(self.src_corpus)

    # idx번째 소스 문장을 토큰 ID 시퀀스로 인코딩, max_len을 넘으면 잘라냄.
    def __getitem__(self, idx):
        src_ids = self.encoder_tokenizer.encode(self.src_corpus[idx])[:self.max_len]
        trg_ids = self.decoder_tokenizer.encode(self.trg_corpus[idx])

        # trg_label에서는 bos(시작 신호) 사용 x
        truncated_trg = trg_ids[:self.max_len-2]
        trg_input = [self.bos_id] + truncated_trg + [self.eos_id]
        trg_label = truncated_trg + [self.eos_id]

        # (목표 길이) - (현재 리스트 길이) 만큼 기존 리스트 뒤에 pad를 이어붙임
        src_ids += [self.pad_id] * (self.max_len - len(src_ids))
        trg_input += [self.pad_id] * (self.max_len - len(trg_input))
        trg_label += [self.pad_id] * (self.max_len - len(trg_label))

        return torch.tensor(src_ids) ,torch.tensor(trg_input), torch.tensor(trg_label)


In [11]:
# 위 내용을 배치 파이프라인으로 연결
# MAX_LEN = 텐서의 가로 길이, src.ids shape(60, )
# 테스트 결과 최대 길이 95, 평균 25.73 나왔으므로 여러가지 고려하여 60으로 결정
MAX_LEN = 60
train_dataset = TranslationDataset(
    ko_corpus, en_corpus,
    encoder_tokenizer, decoder_tokenizer,
    max_len=MAX_LEN,
)

# dataset에서 샘플을 꺼내 배치로 묶어 주는 반복기 (32개씩)
# 32가 무난하다는 ai님의 의견을 적극 반영, 오히려 낮을 때 노이즈가 도움이 되기도 한답니다
# 근데 vocab 6000 기준이라 모르겠음
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


## Step 4. Attentional Seq2seq 모델 설계

기존 Encoder는 attention을 위해 outputs도 반환하도록 되어 있고(원본 그대로),
`Decoder`(옛 버전, attention 없음)는 이제 안 쓰지만 참고용으로 남겨둠.
실제로 학습에 쓰는 건 `AttnDecoder` + `Seq2Seq` 래퍼.

In [12]:
# embed_dim, hidden_dim == 하이퍼파라미터

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id):
        super().__init__()  # 부모 클래스 초기화
        # (6000, ) 테이블, 동시에 학습되는? 가능한 파라미터가 됨 / pad_id 는 항상 0벡터 고정
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        # GRU(RNN 종류), 문장을 한 토큰씩 읽음 -> 내부(hidden state) 업데이트
        # () -> 입력 백터의 차원, (GRU의 내부 기억) hidden state 크기, 차원순서 옵션
        # Pytorch 기본값이 (seq_len, batch, feature)라서
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    # 데이터 넣으면 자동 실행
    def forward(self, src):
        # src : (batch, seq_len)
        # embedding으로 3차원으로 부풀려짐
        # 백터끼리 거리, 유사도 구해야지....
        embedded = self.embedding(src) # -> (batch, seq_len, embed_dim)
        # outputs -> 스텝마다 출력값 전체 반환 / (batch, seq_len, embed_dim)
        # hidden -> 다 읽고 마지막 hidden 하나 반환 / (1, batch, hidden_dim)
        # 1은 레이어 방향 개수, GRU 단방향 사용했음
        # 만약 양방향 GRU 를 써본다면?? --> 그건 미래의 내가
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id):
        super().__init__()  # 똑같음
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        # hidden_dim 차원 백터에서 어떤 토큰일 확률이 높은가 계산
        # hidden_dim 백터를 vocab_size 차원으로 펼침
        self.fc = nn.Linear(hidden_dim, vocab_size) # 행렬곱, 덧셈 / fc = fully connected

        # 전체적으로 encode와 동일
        # 두 번째 인자로 hidden을 넘김
        # -> encoder에서 준 요약 백터를 초기 상태로 삼아서 읽어나감
        # 디코더가 읽은 한국어 문장 내용을 알고 있는 상태에서 영어 생성
    def forward(self, trg_input, hidden):
        embedded = self.embedding(trg_input)
        outputs, hidden = self.rnn(embedded, hidden)
        logits = self.fc(outputs)
        return logits  # 최종 예측 결과(로짓) 반환, 학습 루프에선 특정 코드로 정답과 비교되는 값


In [13]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # [디코더 hidden ; 인코더 output] 을 이어붙인(concat) 걸 hidden_dim 차원으로 한번 눌러줌
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        # 그걸 다시 점수 1개(스칼라)로 눌러줌 -> 이게 "얼마나 이 위치를 볼지" 점수
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        # decoder_hidden : (1, batch, hidden_dim)          <- 디코더 GRU의 "지금" hidden
        # encoder_outputs: (batch, src_len, hidden_dim)    <- 인코더가 만든 타임스텝별 출력 전부
        src_len = encoder_outputs.size(1)

        # decoder_hidden을 src_len만큼 복제해서 encoder_outputs와 나란히 비교 가능하게 만듦
        # (1,batch,hidden) -> (batch,1,hidden) -> (batch,src_len,hidden)
        hidden = decoder_hidden.permute(1, 0, 2).repeat(1, src_len, 1)

        # "지금 디코더 상태"와 "원문의 각 위치"를 나란히 놓고 얼마나 관련있는지 비선형(tanh)으로 계산
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)  # (batch, src_len) - 위치별 관련도 점수(원점수, 아직 확률 아님)

        if mask is not None:
            # 원문에 padding(<pad>)으로 채워둔 자리는 애초에 의미 없는 토큰이라
            # 점수를 -무한대로 깔아뭉개서 softmax 통과 후 가중치가 0이 되게 만듦
            attention = attention.masked_fill(~mask, -1e10)

        # softmax로 "합이 1인 확률분포"로 바꿈 -> 이게 실제 attention 가중치
        return torch.softmax(attention, dim=1)  # (batch, src_len)


# =========================================================
# 2. Attention Decoder
# ---------------------------------------------------------
# 기존 Decoder는 trg_input 문장 전체를 rnn()에 한번에 통째로 넣을 수 있었음
# (매 스텝 필요한 정보가 hidden 하나뿐이라서).
# attention을 쓰면 "매 스텝마다 새로 attention 점수를 계산 -> context 벡터를 뽑아서
# 그 스텝의 입력에 같이 넣어줘야" 하기 때문에, 한 스텝(forward_step)씩 파이썬 for문으로
# 직접 돌려야 함. GRU 자체, Embedding, Linear는 기존 Decoder와 같은 재료.
# =========================================================
class AttnDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)  # 기존과 동일한 용도
        self.attention = Attention(hidden_dim)
        # GRU 입력을 [현재 토큰 임베딩 ; context벡터] 이어붙인 걸로 넣어줌
        # -> "지금 단어" + "원문에서 지금 봐야 할 부분 요약"을 같이 보고 다음 hidden을 만듦
        self.rnn = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        # 최종 예측할 때도 GRU 출력만 보지 않고 context, 방금 넣은 임베딩까지 다 참고
        # (기존 Decoder의 self.fc = nn.Linear(hidden_dim, vocab_size) 을 확장한 버전)
        self.fc = nn.Linear(hidden_dim * 2 + embed_dim, vocab_size)

    def forward_step(self, input_token, hidden, encoder_outputs, mask=None):
        """토큰 딱 1개짜리 스텝만 처리. input_token: (batch, 1)"""
        embedded = self.embedding(input_token)                            # (batch, 1, embed_dim)
        attn_weights = self.attention(hidden, encoder_outputs, mask)      # (batch, src_len) 합=1
        # attn_weights를 가중치 삼아 encoder_outputs를 가중합 -> "이번 스텝에 필요한 원문 요약"
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)   # (batch, 1, hidden_dim)

        rnn_input = torch.cat((embedded, context), dim=2)   # (batch, 1, embed_dim+hidden_dim)
        output, hidden = self.rnn(rnn_input, hidden)         # 기존과 동일한 GRU, 입력 크기만 커짐

        # fc에 넣기 전에 (batch, 1, dim) -> (batch, dim)로 차원 하나 없애줌
        output = output.squeeze(1)
        context = context.squeeze(1)
        embedded = embedded.squeeze(1)
        logits = self.fc(torch.cat((output, context, embedded), dim=1))  # (batch, vocab_size)
        # attn_weights도 같이 반환 -> 나중에 attention map 그릴 때(7번, 보너스) 씀
        return logits, hidden, attn_weights

    def forward(self, trg_input, hidden, encoder_outputs, mask=None, teacher_forcing_ratio=1.0):
        """학습용. trg_input: (batch, trg_len) 전체를 받지만 내부에서 한 스텝씩 처리."""
        batch_size, trg_len = trg_input.size()
        vocab_size = self.fc.out_features
        # 스텝별 예측(logits)을 저장할 그릇을 미리 0으로 만들어둠
        outputs = torch.zeros(batch_size, trg_len, vocab_size, device=trg_input.device)

        input_token = trg_input[:, 0].unsqueeze(1)  # 맨 첫 입력은 <bos> (기존 TranslationDataset이 이미 붙여줌)
        for t in range(trg_len):
            logits, hidden, _ = self.forward_step(input_token, hidden, encoder_outputs, mask)
            outputs[:, t] = logits
            if t + 1 < trg_len:
                # teacher forcing: 정답 라벨을 다음 입력으로 쓸지, 모델이 방금 예측한 단어를 쓸지
                # 확률적으로 결정 (초반엔 정답을 많이 보여줘야 학습이 안정적으로 됨)
                use_teacher = random.random() < teacher_forcing_ratio
                input_token = trg_input[:, t + 1].unsqueeze(1) if use_teacher else logits.argmax(1).unsqueeze(1)
        return outputs


# =========================================================
# 3. Encoder + AttnDecoder를 하나로 묶는 래퍼
# ---------------------------------------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_id = src_pad_id  # attention이 padding 위치를 가려내기 위해 필요

    def forward(self, src, trg_input, teacher_forcing_ratio=0.5):
        encoder_outputs, hidden = self.encoder(src)
        mask = (src != self.src_pad_id)  # (batch, src_len) - True면 진짜 토큰, False면 padding
        outputs = self.decoder(trg_input, hidden, encoder_outputs, mask, teacher_forcing_ratio)
        return outputs


In [14]:
# =========================================================
# 4. Step 4 - 하이퍼파라미터 (Embedding Size / Hidden Size) 실험
# ---------------------------------------------------------
#   EMBED_DIM=128, HIDDEN_DIM=256   -> 가볍고 빠름, 베이스라인 확인용
#   EMBED_DIM=256, HIDDEN_DIM=512   -> 무난한 기본값 (아래 기본 선택)
#   EMBED_DIM=512, HIDDEN_DIM=512   -> 임베딩을 더 키운 버전, 어휘 표현력 ↑
# =========================================================
EMBED_DIM = 256
HIDDEN_DIM = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)
# GPU가 실제로 잡히는지 바로 확인하고 싶다면 아래 주석 풀고 실행
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU name:", torch.cuda.get_device_name(0))

encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, encoder_tokenizer.pad_id()).to(device)
decoder = AttnDecoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, decoder_tokenizer.pad_id()).to(device)
model = Seq2Seq(encoder, decoder, encoder_tokenizer.pad_id()).to(device)

# optimizer/loss는 기존 코드 스타일과 동일한 재료(Adam, CrossEntropyLoss)만 사용
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# pad 위치는 애초에 맞고 틀리고를 따질 필요가 없는 자리라 loss 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=decoder_tokenizer.pad_id())


device: cuda


## Step 5. 번역 함수 (추론용, greedy decoding)

In [15]:
def translate_sentence(model, sentence, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN):
    model.eval()  # dropout 등 없어서 지금 구조엔 큰 차이 없지만, 습관적으로 평가 모드로 전환
    src_ids = encoder_tokenizer.encode(preprocess_korean(sentence))[:max_len]
    src_ids = src_ids + [encoder_tokenizer.pad_id()] * (max_len - len(src_ids))  # 기존 Dataset과 같은 padding 방식
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)  # (1, max_len) - 문장 1개짜리 배치

    with torch.no_grad():  # 추론이라 기울기 계산 불필요
        encoder_outputs, hidden = model.encoder(src_tensor)
        mask = (src_tensor != encoder_tokenizer.pad_id())

        input_token = torch.tensor([[decoder_tokenizer.bos_id()]], device=device)  # 첫 입력은 <bos>
        result_ids = []
        attn_history = []  # 보너스(7번)에서 attention map 그릴 때 쓸 기록
        for _ in range(max_len):
            logits, hidden, attn_weights = model.decoder.forward_step(input_token, hidden, encoder_outputs, mask)
            top1 = logits.argmax(1).item()  # 가장 확률 높은 단어 하나만 고르는 greedy decoding
            if top1 == decoder_tokenizer.eos_id():
                break  # <eos> 나오면 문장 끝
            result_ids.append(top1)
            attn_history.append(attn_weights.squeeze(0).cpu())
            input_token = torch.tensor([[top1]], device=device)  # 방금 예측한 단어를 다음 입력으로

    model.train()  # 학습 루프 중간에 호출해도 이어서 학습 모드로 되돌아가도록
    # result_ids도 같이 반환 -> attention map 그릴 때 decode(문자열)를 다시 encode해서
    # 토큰을 재구성하면 SentencePiece 특성상 원래 id와 안 맞을 수 있어서, 생성 시점의
    # id를 그대로 넘겨줘서 그 문제를 피함
    return decoder_tokenizer.decode(result_ids), attn_history, result_ids


## Step 5 (계속). 학습 루프

`eval_step()`은 별도로 만들지 않고, 매 epoch이 끝날 때마다 K1~K4 예문을 바로 번역해서 확인합니다.
학습이 끝난 뒤 콘솔에 쌓인 epoch별 번역 중 가장 그럴듯한 것을 골라 E1~E4로 제출하면 됩니다.

In [ ]:
# =========================================================
# 6. 학습 루프 (Step 5)
# ---------------------------------------------------------
# eval_step() 같은 별도 검증 함수는 만들지 않음(과제 안내에 따름).
# 대신 매 epoch이 끝날 때마다 바로 K1~K4를 번역해서 화면에 찍어보고,
# 그 중 제일 마음에 드는(=E1~E4로 제출할) epoch을 사람이 직접 골라내는 방식.
# train_loader, MAX_LEN, VOCAB_SIZE 등은 전부 기존 코드에서 이미 만들어둔 것을 그대로 재사용.
# =========================================================
N_EPOCHS = 20
CLIP = 1.0                     # GRU 계열은 gradient가 튀는 경우가 있어 클리핑으로 학습 안정화
TEACHER_FORCING_RATIO = 0.5
CHECKPOINT_PATH = os.path.join(seq2seq_dir, 'checkpoint.pt')  # epoch마다 저장 → 중단해도 이어서 학습 가능

# 과제에서 준 4개 예문 (K1~K4). 그대로 번역 확인용으로 사용
example_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다.",
]

# 체크포인트가 있으면 마지막 완료 epoch 다음부터 재개
start_epoch = 1
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f"✅ 체크포인트 로드: Epoch {ckpt['epoch']} (loss {ckpt['avg_loss']:.4f}) → Epoch {start_epoch}부터 재개")

for epoch in range(start_epoch, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for src, trg_input, trg_label in train_loader:   # 기존 train_loader 그대로 사용
        src = src.to(device)
        trg_input = trg_input.to(device)
        trg_label = trg_label.to(device)

        optimizer.zero_grad()
        outputs = model(src, trg_input, teacher_forcing_ratio=TEACHER_FORCING_RATIO)  # (batch, trg_len, vocab)

        # CrossEntropyLoss는 (N, class) vs (N,) 형태를 기대하므로 배치*시퀀스를 한 줄로 펼침
        loss = criterion(outputs.reshape(-1, outputs.size(-1)), trg_label.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"\n[Epoch {epoch:02d}/{N_EPOCHS}] avg loss: {avg_loss:.4f}")

    # ---- 매 epoch 끝날 때마다 K1~K4 예문을 바로 번역해서 확인 (eval_step 없이 인라인으로) ----
    for i, sent in enumerate(example_sentences, start=1):
        pred, _, _ = translate_sentence(model, sent, encoder_tokenizer, decoder_tokenizer)
        print(f"  K{i}) {sent}  ->  {pred}")

    # epoch 하나 끝날 때마다 체크포인트 저장 (중간에 멈춰도 마지막 완료 epoch부터 재개 가능)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'avg_loss': avg_loss,
    }, CHECKPOINT_PATH)
    print(f"  💾 checkpoint saved → {CHECKPOINT_PATH}")

# 학습이 다 끝난 뒤, 위에서 epoch별로 찍힌 K1~K4 번역 결과 중
# 본인이 보기에 가장 그럴듯한 epoch의 결과를 그대로 E1~E4로 제출하면 됨.


# 학습 결과 날려먹음 
# 총 180m 4회 나옴 그래도 4회쯤 점점 잡아갔었음
# ex) 일곱 명의 사망자가 발생했다. -> 7 was people killed.

KeyboardInterrupt: 

## (보너스, 선택) Attention Map 시각화

In [ ]:
# =========================================================
# 7. (보너스, 선택) Attention Map 시각화
# =========================================================
import matplotlib.pyplot as plt

def plot_attention(model, sentence, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN):
    pred, attn_history, result_ids = translate_sentence(model, sentence, encoder_tokenizer, decoder_tokenizer, max_len)

    src_ids = encoder_tokenizer.encode(preprocess_korean(sentence))[:max_len]
    src_tokens = [encoder_tokenizer.id_to_piece(i) for i in src_ids]
    # decode한 문자열을 다시 encode하지 않고, 생성 시점의 result_ids를 그대로 토큰화
    trg_tokens = [decoder_tokenizer.id_to_piece(i) for i in result_ids]

    # attn_history: 생성된 영어 토큰 개수 x src_len 형태로 쌓기 (길이가 정확히 일치함)
    attn_matrix = torch.stack(attn_history).numpy()

    fig, ax = plt.subplots(figsize=(max(4, len(src_tokens) * 0.6), max(3, len(trg_tokens) * 0.6)))
    im = ax.imshow(attn_matrix, cmap='viridis')
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, rotation=90)
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_yticklabels(trg_tokens)
    ax.set_xlabel("입력(한국어) 토큰")
    ax.set_ylabel("생성된 영어 토큰")
    fig.colorbar(im, ax=ax)
    plt.title(f"Attention map: {sentence}")
    plt.tight_layout()
    plt.savefig("attention_map_sample.png")  # 한글이 네모(□)로 깨지면 시스템에 맞는 한글 폰트를 plt.rcParams['font.family']에 지정해야 함
    plt.show()
    print("attention_map_sample.png 로 저장됨")

# 사용 예 (필요할 때 주석 풀고 실행):
# plot_attention(model, "오바마는 대통령이다.", encoder_tokenizer, decoder_tokenizer)


---

## 소감 및 각주

### 이번 실습의 의의
- **Seq2Seq + Attention** 구조를 직접 구현해 보면서, "인코더가 문장을 읽고 → 디코더가 단어를 하나씩 생성한다"는 흐름을 코드 단위로 이해할 수 있었다.
- SentencePiece 토크나이저, `Dataset`/`DataLoader`, teacher forcing, greedy decoding까지 **번역기 파이프라인 전체**를 한 번에 경험한 점이 가장 큰 수확이다.
- K1~K4 예문을 epoch마다 출력해 보며, loss 숫자만큼이나 **실제 번역 품질 변화**를 눈으로 확인하는 습관이 생겼다.

### 학습 과정에서 겪은 점
- epoch 1회당 시간이 생각보다 길어서, 20 epoch를 끝까지 돌리는 데 **상당한 인내가 필요**했다. (특히 CPU 환경이었다면 더 힘들었을 것)
- 중간에 학습을 멈추면(Interrupt / 커널 재시작) **모델 가중치가 전부 사라지는** 문제가 있어, `torch.save` 기반 **체크포인트 기능**을 직접 추가했다.
  - 매 epoch 종료 시 `./Seq2seq/checkpoint.pt`에 저장
  - 재실행 시 마지막 완료 epoch 다음부터 자동 재개
- 이 경험을 통해 "학습 코드 = forward/backward만이 아니라, **중단·재개를 고려한 설계**도 필요하다"는 걸 체감했다.

### 추가로 해보고 싶은 탐구 (의욕 메모)
1. **하이퍼파라미터 비교** — `EMBED_DIM`/`HIDDEN_DIM`, `batch_size`, `TEACHER_FORCING_RATIO`를 바꿔가며 K1~K4 번역 품질과 loss 곡선 비교
2. **Beam Search** — greedy decoding 대신 beam search를 적용하면 반복·누락(예: `coffee coffee`)이 줄어드는지 확인
3. **Attention Map 분석** — 어떤 한국어 토큰에 디코더가 집중하는지 시각화해 보며, 오역 패턴과 연결지어 해석
4. **양방향 Encoder(BiGRU)** — 단방향 GRU vs 양방향 GRU 성능 차이 실험
5. **Transformer로 확장** — RNN 기반 Seq2Seq와 Transformer 번역기의 속도·품질 비교 (시간이 된다면!)

> *"번역 품질이 아직 완벽하진 않지만, epoch를 돌릴수록 점점 나아지는 걸 직접 확인한 경험 자체가 다음 NLP 과제로 이어지는 동력이 될 것 같다."*